In [27]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.join(data["python_files"])
pathway = os.path.join(data["publications"], "halina2018goal")
original_data_pathway = os.path.join(pathway, "original_data")
complete_path_1 = os.path.join(original_data_pathway, "halina_2018_S1Table.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [28]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)

df['study_id']="halina2018goal"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
# df.columns


In [29]:
df.rename(columns={"subject": "ape", 
    "start time (s)":"start_time_s", 
    "end time (s)":"end_time_s",
    "point duration (s)":"point_duration_s",
    "grape position":"grape_position",
    "correct?":"correct",
    "object received":"object_received",
    "eye contact": "eye_contact",
    "point direction": "point_direction"}, inplace=True)

In [30]:
code_list=["condition"]
for index, x in enumerate(code_list):    
    df[x] = df[x].astype(str)
    temp=[]
    for entry in df[x]:
        if entry == 'f':
            entry = "failed_look"
        elif entry =='s':
            entry = "successful_look"
        elif entry =='m':
            entry = "motivational"
        temp.append(entry)
    df = df.assign(temp_col=temp)
    df=df.rename(columns={'temp_col': x+'_codes'})

In [31]:
code_list=["hand"]
for index, x in enumerate(code_list):    
    df[x] = df[x].astype(str)
    temp=[]
    for entry in df[x]:
        if entry == 'l':
            entry = "left_handed_point"
        elif entry =='r':
            entry = "right_handed_point"
        temp.append(entry)
    df = df.assign(temp_col=temp)
    df=df.rename(columns={'temp_col': x+'_codes'})

In [32]:
code_list=["eye_contact"]
for index, x in enumerate(code_list):    
    df[x] = df[x].astype(str)
    temp=[]
    for entry in df[x]:
        if entry == 'n':
            entry = "no_eye_contact_during_point"
        elif entry =='y':
            entry = "eye_contact_during_point"
        temp.append(entry)
    df = df.assign(temp_col=temp)
    df=df.rename(columns={'temp_col': x+'_codes'})

In [33]:
code_list=["point_direction"]
for index, x in enumerate(code_list):    
    df[x] = df[x].astype(str)
    temp=[]
    for entry in df[x]:
        if entry == 'l':
            entry = "point_directed_left"
        elif entry =='c':
            entry = "point_directed_to_center"
        elif entry == "r":
            entry = "point_directed_right"
        temp.append(entry)
    df = df.assign(temp_col=temp)
    df=df.rename(columns={'temp_col': x+'_codes'})

In [34]:
code_list=["grape_position"]
for index, x in enumerate(code_list):    
    df[x] = df[x].astype(str)
    temp=[]
    for entry in df[x]:
        if entry == 'l':
            entry = "left"
        elif entry =='r':
            entry = "right"
        temp.append(entry)
    df = df.assign(temp_col=temp)
    df=df.rename(columns={'temp_col': x+'_codes'})

In [35]:
code_list=["correct"]
for index, x in enumerate(code_list):    
    df[x] = df[x].astype(str)
    temp=[]
    for entry in df[x]:
        if entry == 'c':
            entry = "point_directed_at_grape"
        elif entry =='inc':
            entry = "point_not_directed_at_grape"
        temp.append(entry)
    df = df.assign(temp_col=temp)
    df=df.rename(columns={'temp_col': x+'_codes'})

In [36]:
code_list=["object_received"]
for index, x in enumerate(code_list):    
    df[x] = df[x].astype(str)
    temp=[]
    for entry in df[x]:
        if entry == '0' or entry == '0.0':
            entry = "no_reward"
        elif entry =='1' or entry == '1.0':
            entry = "grape_received"
        elif entry =='2' or entry == '2.0':
            entry = "grapeless_grapevine_received"
        temp.append(entry)
    df = df.assign(temp_col=temp)
    df=df.rename(columns={'temp_col': x+'_codes'})

In [37]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='ape', right_on='name', how='left')

In [38]:
# df.columns
df.rename(columns={"ape": "participant",
                   "start_time_s":"start_time_in_seconds",
                   "end_time_s":"end_time_in_seconds",
                   "point_duration_s":"point_duration_in_seconds"}, inplace=True)

In [39]:
halina2018goal_standardized=df[['study_id', 'participant', 'sex', 'species', 'session', 'trial', 'condition_codes',
       'start_time_in_seconds', 'end_time_in_seconds',
       'point_duration_in_seconds', 'hand_codes', 'eye_contact_codes', 'point_direction_codes',
        'grape_position_codes','correct_codes', 
        'object_received_codes','experimenter'  ]]

halina2018goal_standardized.columns =halina2018goal_standardized.columns.str.replace('_codes', '')

comp_out_path_stand = os.path.join(out_pathway, 'halina2018goal_standardized.csv')
halina2018goal_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =halina2018goal_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
halina2018goal_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'halina2018goal_glossary.csv')
halina2018goal_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
